# Limpieza y estandarización de Subsidios Educativos

Este notebook toma el archivo `Excel_Origen_Desordenado_Subsidios.xlsx`, limpia encabezados, elimina filas basura, corrige inconsistencias y genera `subsidios_educativos.csv`.

In [ ]:
import pandas as pd
import numpy as np

archivo = "Excel_Origen_Subsidios.xlsx"
xls = pd.ExcelFile(archivo)

print(xls.sheet_names)

['BASE_FINAL_V3', 'Catalogos', 'Presupuesto', 'Notas']


In [3]:
# La hoja principal contiene títulos y notas antes de los datos
raw = pd.read_excel(
    archivo,
    sheet_name="BASE_FINAL_V3",
    header=5
)

raw.head()

,ID SOLICITUD,FECHA SOLICITUD,departamento,zona,TIPO BENEFICIO,canal,NIVEL EDUCATIVO TUTOR,NUM PERSONAS HOGAR,ingreso hogar,puntaje focalizacion,...,GESTOR ASIGNADO,tiempo ciclo dias,FECHA DECISION,FECHA ENTREGA,COSTO ATENCION,COSTO BENEFICIO,permanencia escolar,TRANSITO EDUCACION SUPERIOR,INGRESO HOGAR POSTERIOR,SATISFACCION SERVICIO
0,SOL-00001,2025-12-04,BolÃ­var,Rural,Plan_movil,Presencial,Secundaria,5.0,1166000.0,51.8,...,Gestor_05,12.0,2025-12-16,2025-12-21,36900.0,255600.0,0.0,0.0,1279000.0,5.0
1,SOL-00002,2025-05-23,Norte de Santander,Rural,Biblioteca_digital,Presencial,Secundaria,5.0,1076000.0,71.5,...,Gestor_14,12.0,2025-06-04,2025-06-04,26900.0,49800.0,0.0,0.0,NaN,2.0
2,SOL-00003,2026-04-09,Sucre,Urbano,Computador,Presencial,Tecnico,3.0,1309000.0,42.3,...,Gestor_04,14.0,2026-04-23,NaN,27900.0,NaN,0.0,0.0,1428000.0,NaN
3,SOL-00004,2025-10-14,Cesar,Rural,Internet,Digital,Secundaria,6.0,1180000.0,54.3,...,Gestor_05,7.0,2025-10-21,2025-10-22,9600.0,346600.0,1.0,0.0,1254000.0,3.0
4,SOL-00005,2025-07-24,Atlántico,Urbano,Computador,Presencial,Tecnico,2.0,1544000.0,40.2,...,Gestor_04,21.0,2025-08-14,2025-08-19,23900.0,972700.0,1.0,NaN,1636000.0,3.0


In [4]:
# Normalización de nombres de columnas
raw.columns = (
    raw.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

raw.columns

Index(['id_solicitud', 'fecha_solicitud', 'departamento', 'zona',
       'tipo_beneficio', 'canal', 'nivel_educativo_tutor',
       'num_personas_hogar', 'ingreso_hogar', 'puntaje_focalizacion',
       'estado_solicitud', 'gestor_asignado', 'tiempo_ciclo_dias',
       'fecha_decision', 'fecha_entrega', 'costo_atencion', 'costo_beneficio',
       'permanencia_escolar', 'transito_educacion_superior',
       'ingreso_hogar_posterior', 'satisfaccion_servicio'],
      dtype='object')

In [5]:
# Eliminar filas completamente vacías
df = raw.dropna(how="all").copy()

# Corrección de codificaciones frecuentes
reemplazos = {
    "BolÃ­var": "Bolívar",
    "ATLÁNTICO": "Atlántico",
    "atlántico": "Atlántico",
    "sucre": "Sucre",
    "CÓRDOBA": "Córdoba"
}

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].replace(reemplazos)

df.head()

,id_solicitud,fecha_solicitud,departamento,zona,tipo_beneficio,canal,nivel_educativo_tutor,num_personas_hogar,ingreso_hogar,puntaje_focalizacion,...,gestor_asignado,tiempo_ciclo_dias,fecha_decision,fecha_entrega,costo_atencion,costo_beneficio,permanencia_escolar,transito_educacion_superior,ingreso_hogar_posterior,satisfaccion_servicio
0,SOL-00001,2025-12-04,Bolívar,Rural,Plan_movil,Presencial,Secundaria,5.0,1166000.0,51.8,...,Gestor_05,12.0,2025-12-16,2025-12-21,36900.0,255600.0,0.0,0.0,1279000.0,5.0
1,SOL-00002,2025-05-23,Norte de Santander,Rural,Biblioteca_digital,Presencial,Secundaria,5.0,1076000.0,71.5,...,Gestor_14,12.0,2025-06-04,2025-06-04,26900.0,49800.0,0.0,0.0,NaN,2.0
2,SOL-00003,2026-04-09,Sucre,Urbano,Computador,Presencial,Tecnico,3.0,1309000.0,42.3,...,Gestor_04,14.0,2026-04-23,NaN,27900.0,NaN,0.0,0.0,1428000.0,NaN
3,SOL-00004,2025-10-14,Cesar,Rural,Internet,Digital,Secundaria,6.0,1180000.0,54.3,...,Gestor_05,7.0,2025-10-21,2025-10-22,9600.0,346600.0,1.0,0.0,1254000.0,3.0
4,SOL-00005,2025-07-24,Atlántico,Urbano,Computador,Presencial,Tecnico,2.0,1544000.0,40.2,...,Gestor_04,21.0,2025-08-14,2025-08-19,23900.0,972700.0,1.0,NaN,1636000.0,3.0


In [6]:
# Conversión de fechas
for c in ["fecha_solicitud","fecha_decision","fecha_entrega"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

# Conversión numérica
for c in ["costo_atencion","costo_beneficio","puntaje_focalizacion"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2500 entries, 0 to 2534
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   id_solicitud                 2500 non-null   object        
 1   fecha_solicitud              2500 non-null   datetime64[ns]
 2   departamento                 2500 non-null   object        
 3   zona                         2500 non-null   object        
 4   tipo_beneficio               2500 non-null   object        
 5   canal                        2500 non-null   object        
 6   nivel_educativo_tutor        2500 non-null   object        
 7   num_personas_hogar           2500 non-null   float64       
 8   ingreso_hogar                2500 non-null   float64       
 9   puntaje_focalizacion         2500 non-null   float64       
 10  estado_solicitud             2500 non-null   object        
 11  gestor_asignado              2500 non-null   obj

In [7]:
# Eliminar duplicados potenciales
if "id_solicitud" in df.columns:
    df = df.drop_duplicates(subset=["id_solicitud"])

print("Registros finales:", len(df))

Registros finales: 2500


In [8]:
# Regla financiera de auditoría:
# solicitudes rechazadas no pueden tener desembolso

if "estado_solicitud" in df.columns and "costo_beneficio" in df.columns:
    df.loc[
        df["estado_solicitud"].astype(str).str.upper() == "RECHAZADA",
        "costo_beneficio"
    ] = 0

df.head()

,id_solicitud,fecha_solicitud,departamento,zona,tipo_beneficio,canal,nivel_educativo_tutor,num_personas_hogar,ingreso_hogar,puntaje_focalizacion,...,gestor_asignado,tiempo_ciclo_dias,fecha_decision,fecha_entrega,costo_atencion,costo_beneficio,permanencia_escolar,transito_educacion_superior,ingreso_hogar_posterior,satisfaccion_servicio
0,SOL-00001,2025-12-04,Bolívar,Rural,Plan_movil,Presencial,Secundaria,5.0,1166000.0,51.8,...,Gestor_05,12.0,2025-12-16,2025-12-21,36900.0,255600.0,0.0,0.0,1279000.0,5.0
1,SOL-00002,2025-05-23,Norte de Santander,Rural,Biblioteca_digital,Presencial,Secundaria,5.0,1076000.0,71.5,...,Gestor_14,12.0,2025-06-04,2025-06-04,26900.0,49800.0,0.0,0.0,NaN,2.0
2,SOL-00003,2026-04-09,Sucre,Urbano,Computador,Presencial,Tecnico,3.0,1309000.0,42.3,...,Gestor_04,14.0,2026-04-23,NaT,27900.0,0.0,0.0,0.0,1428000.0,NaN
3,SOL-00004,2025-10-14,Cesar,Rural,Internet,Digital,Secundaria,6.0,1180000.0,54.3,...,Gestor_05,7.0,2025-10-21,2025-10-22,9600.0,346600.0,1.0,0.0,1254000.0,3.0
4,SOL-00005,2025-07-24,Atlántico,Urbano,Computador,Presencial,Tecnico,2.0,1544000.0,40.2,...,Gestor_04,21.0,2025-08-14,2025-08-19,23900.0,972700.0,1.0,NaN,1636000.0,3.0


In [9]:
# Exportación final

df.to_csv("subsidios_educativos_v2.csv", index=False, encoding="utf-8")

print("Archivo generado: subsidios_educativos_v2.csv")

Archivo generado: subsidios_educativos_v2.csv
